# 01. Revisión inicial de los datos

En este primer paso vamos a comprobar que los cinco archivos se puedan leer y relacionar correctamente. Revisaremos su estructura, fechas, valores faltantes, duplicados y continuidad horaria.

**Alcance:** esta es una revisión de integridad. Todavía no vamos a limpiar archivos, generar variables, construir etiquetas, entrenar modelos ni preparar el reporte. Las revisiones de estructura abarcan los archivos originales; no usamos los resultados del periodo de prueba para explorar patrones o elegir decisiones del modelo.

## 1. Preparar las herramientas y encontrar los archivos

Solo necesitamos pandas para manejar las tablas y NumPy para comprobar los valores numéricos. `Path` nos ayuda a encontrar la carpeta `Datos` sin escribir una ruta que dependa de una computadora.

El notebook puede ejecutarse desde la raíz del repositorio o desde `Preprocesamiento`. No necesitamos fijar una semilla porque en esta revisión no hacemos operaciones aleatorias.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Buscar Datos en la carpeta actual o en su carpeta superior.

# Obtener la carpeta desde la que se está ejecutando el notebook.
carpeta_actual = Path.cwd()

# Si Datos está aquí, ya estamos en la raíz del repositorio.
if (carpeta_actual / "Datos").is_dir():
    ruta_repo = carpeta_actual
    
# Si estamos en una subcarpeta del proyecto, buscar Datos un nivel arriba.
elif (carpeta_actual.parent / "Datos").is_dir():
    ruta_repo = carpeta_actual.parent
else:
    raise FileNotFoundError("Abre el notebook desde el repositorio o desde Preprocesamiento.")

# Construir la ruta de los CSV sin imprimir rutas personales.
ruta_datos = ruta_repo / "Datos"

# Estas opciones solo cambian cómo se muestran las tablas, no sus valores.
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)

## 2. Cargar los cinco archivos

Guardaremos cada tabla por separado. Primero comprobaremos que todos los archivos estén disponibles y después mostraremos cuántas filas y columnas contiene cada uno.

In [2]:
# Relacionar un nombre corto con el nombre real de cada archivo CSV.
archivos = {
    "telemetria": "PdM_telemetry.csv",
    "errores": "PdM_errors.csv",
    "mantenimiento": "PdM_maint.csv",
    "fallas": "PdM_failures.csv",
    "maquinas": "PdM_machines.csv",
}

# Aquí guardaremos cada tabla usando su nombre corto como clave.
tablas = {}

# Acumular un resumen por archivo antes de convertirlo en una tabla.
resumen_archivos = []

# Recorrer el diccionario: nombre es la etiqueta y archivo es el CSV correspondiente.
for nombre, archivo in archivos.items():
    
    # Combinar la carpeta Datos con el nombre del archivo de esta vuelta.
    ruta_archivo = ruta_datos / archivo
    
    # Detenernos con un mensaje claro si el archivo no existe.
    if not ruta_archivo.is_file():
        raise FileNotFoundError(f"No se encontró: {ruta_archivo}")
    
    # Leer el CSV; por ahora pandas identifica los tipos automáticamente.
    tabla = pd.read_csv(ruta_archivo)
    
    # Guardar la tabla para reutilizarla en las siguientes secciones.
    tablas[nombre] = tabla
    
    # Agregar una fila de resumen. 
    resumen_archivos.append({
        "tabla": nombre,
        "archivo": archivo,
        "filas": len(tabla),
        "columnas": len(tabla.columns),
        # Dividir entre 1024 al cuadrado convierte bytes a MB; round limita los decimales mostrados.
        "tamaño_MB": round(ruta_archivo.stat().st_size / (1024 ** 2), 2),
    })

# Convertir la lista de resúmenes en una tabla visible.
display(pd.DataFrame(resumen_archivos))

,tabla,archivo,filas,columnas,tamaño_MB
0,telemetria,PdM_telemetry.csv,876100,6,76.43
1,errores,PdM_errors.csv,3919,3,0.12
2,mantenimiento,PdM_maint.csv,3286,3,0.10
3,fallas,PdM_failures.csv,761,3,0.02
4,maquinas,PdM_machines.csv,100,3,0.00


Los archivos contienen **876,100 registros de telemetría, 3,919 errores, 3,286 mantenimientos, 761 registros de falla y 100 máquinas**. La telemetría es la tabla más grande porque registra las mediciones por hora. Un registro de falla representa un componente; varios componentes pueden fallar en una misma máquina y hora.

## 3. Revisar columnas y tipos de datos

Comprobaremos que cada tabla tenga las columnas esperadas. También veremos cómo las interpretó pandas al leer los archivos. Es normal que `datetime` se lea inicialmente como texto; en la siguiente sección la convertiremos a fecha y hora.

In [3]:
# Definir la estructura que debe tener cada fuente para poder relacionarla después.
columnas_esperadas = {
    "telemetria": ["datetime", "machineID", "volt", "rotate", "pressure", "vibration"],
    "errores": ["datetime", "machineID", "errorID"],
    "mantenimiento": ["datetime", "machineID", "comp"],
    "fallas": ["datetime", "machineID", "failure"],
    "maquinas": ["machineID", "model", "age"],
}

revision_columnas = []
tipos_originales = []
for nombre, tabla in tablas.items():
    
    # Un conjunto permite comparar nombres de columnas sin depender de su orden.
    esperadas = set(columnas_esperadas[nombre])
    encontradas = set(tabla.columns)
    
    # La resta de conjuntos identifica columnas faltantes o adicionales; sorted ordena sus nombres.
    revision_columnas.append({
        "tabla": nombre,
        "columnas_faltantes": sorted(esperadas - encontradas),
        "columnas_extra": sorted(encontradas - esperadas),
    })
    
    # Recorrer cada columna para registrar el tipo que pandas le asignó al cargarla.
    for columna in tabla.columns:
        tipos_originales.append({
            "tabla": nombre,
            "columna": columna,
            # dtype describe el tipo de la columna; str lo convierte a texto para mostrarlo.
            "tipo_al_cargar": str(tabla[columna].dtype),
        })

display(pd.DataFrame(revision_columnas))
display(pd.DataFrame(tipos_originales))

# Detener la ejecución si falta una columna necesaria.
for nombre, tabla in tablas.items():
    
    # issubset comprueba que estén presentes todas las columnas necesarias.
    if not set(columnas_esperadas[nombre]).issubset(tabla.columns):
        raise ValueError(f"Faltan columnas necesarias en {nombre}.")

,tabla,columnas_faltantes,columnas_extra
0,telemetria,[],[]
1,errores,[],[]
2,mantenimiento,[],[]
3,fallas,[],[]
4,maquinas,[],[]


,tabla,columna,tipo_al_cargar
0,telemetria,datetime,str
1,telemetria,machineID,int64
2,telemetria,volt,float64
3,telemetria,rotate,float64
4,telemetria,pressure,float64
5,telemetria,vibration,float64
6,errores,datetime,str
7,errores,machineID,int64
8,errores,errorID,str
9,mantenimiento,datetime,str


Las cinco tablas tienen las columnas esperadas, sin columnas adicionales. Los identificadores de máquina y la antigüedad son enteros; los cuatro sensores son numéricos. Los tipos de errores, componentes y modelos se leen como texto.

## 4. Revisar fechas y cobertura de los archivos

Convertiremos las fechas en las tablas que están en memoria. Si una fecha no puede interpretarse, pandas la marcará como `NaT` y la contaremos. Además, comprobaremos si los eventos están registrados en horas exactas.

Este cambio de tipo se hace solo en memoria: los CSV originales permanecen intactos.

In [4]:
# El catálogo de máquinas no tiene fecha, así que no se incluye en esta conversión.
tablas_temporales = ["telemetria", "errores", "mantenimiento", "fallas"]
resumen_fechas = []
for nombre in tablas_temporales:
    tabla = tablas[nombre]
    
    # Convertir el texto al formato año-mes-día hora:minuto:segundo; coerce marca fechas inválidas como NaT.
    fechas = pd.to_datetime(
        tabla["datetime"],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )
    
    # Actualizar únicamente la tabla en memoria, sin escribir sobre el CSV.
    tablas[nombre]["datetime"] = fechas
    
    # Excluir fechas inválidas solo de la comprobación de horas exactas; se cuentan por separado.
    fechas_validas = fechas.dropna()
    
    # floor lleva cada fecha al inicio de su hora; si cambia, había minutos o segundos adicionales.
    fuera_de_hora = fechas_validas != fechas_validas.dt.floor("h")
    resumen_fechas.append({
        "tabla": nombre,
        "primera_fecha": fechas.min(),
        "ultima_fecha": fechas.max(),
        "fechas_invalidas_o_vacias": int(fechas.isna().sum()),
        "registros_fuera_de_hora_exacta": int(fuera_de_hora.sum()),
    })

display(pd.DataFrame(resumen_fechas))

# any detecta si al menos una tabla tiene fechas inválidas y detiene los pasos posteriores.
if any(fila["fechas_invalidas_o_vacias"] > 0 for fila in resumen_fechas):
    raise ValueError("Hay fechas que deben revisarse antes de continuar.")

,tabla,primera_fecha,ultima_fecha,fechas_invalidas_o_vacias,registros_fuera_de_hora_exacta
0,telemetria,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0
1,errores,2015-01-01 06:00:00,2016-01-01 05:00:00,0,0
2,mantenimiento,2014-06-01 06:00:00,2016-01-01 06:00:00,0,0
3,fallas,2015-01-02 03:00:00,2015-12-31 06:00:00,0,0


Todas las fechas se pudieron convertir y están registradas en horas exactas.

La telemetría comienza el **1 de enero de 2015 a las 06:00** y termina el **1 de enero de 2016 a las 06:00**. Los mantenimientos comienzan en junio de 2014 y llegan hasta enero de 2016. Esto requiere cuidado: no debemos suponer que todos los archivos contienen únicamente fechas de 2015, ni cambiar sus horas sin una razón documentada. Los archivos no indican una zona horaria.

Los mantenimientos de 2014 podrían aportar historia anterior al inicio de la telemetría. Su uso se decidirá al analizar los reemplazos; en este paso se conservan.

### Recordatorio de la partición obligatoria

Usaremos la siguiente partición temporal, que se aplicará al construir las etiquetas:

| Conjunto | Horas a predecir |
|---|---|
| Entrenamiento | 1 de enero a 31 de agosto de 2015 |
| Validación | 1 de septiembre a 31 de octubre de 2015 |
| Prueba | 1 de noviembre a 31 de diciembre de 2015 |

El horizonte será **24 horas**, con etiqueta definida sobre **(t, t + 24 h]**. Habrá que dejar ese margen en las fronteras: una etiqueta de entrenamiento no puede depender de septiembre y una de validación no puede depender de noviembre. La historia de entrada sí puede comenzar antes de cada periodo.

Todavía no generamos ni exportamos conjuntos. Los registros de enero de 2016 no amplían los periodos establecidos; su tratamiento y el seguimiento disponible al final del año se documentarán al construir las etiquetas. La prueba queda reservada para la semana 3.

## 5. Buscar valores faltantes y números no válidos

Revisaremos todas las columnas para detectar valores vacíos. En las columnas numéricas también buscaremos infinitos y valores que no se puedan convertir a números. No sustituiremos ni eliminaremos datos automáticamente.

In [5]:
# Preparar un resumen por columna para no ocultar dónde faltan datos.
revision_faltantes = []
for nombre, tabla in tablas.items():
    
    # Recorrer cada columna para registrar el tipo que pandas le asignó al cargarla.
    for columna in tabla.columns:
        
        # isna marca los vacíos y sum cuenta cuántos hay; int presenta el conteo como entero.
        cantidad = int(tabla[columna].isna().sum())
        revision_faltantes.append({
            "tabla": nombre,
            "columna": columna,
            "faltantes": cantidad,
            "porcentaje": round(100 * cantidad / len(tabla), 4),
        })
display(pd.DataFrame(revision_faltantes))

# Indicar cuáles columnas deberían contener números en cada tabla.
columnas_numericas = {
    "telemetria": ["machineID", "volt", "rotate", "pressure", "vibration"],
    "errores": ["machineID"],
    "mantenimiento": ["machineID"],
    "fallas": ["machineID"],
    "maquinas": ["machineID", "age"],
}
revision_numerica = []
for nombre, columnas in columnas_numericas.items():
    for columna in columnas:
        
        # Intentar convertir a número; lo que no se pueda interpretar se marca como vacío.
        valores = pd.to_numeric(tablas[nombre][columna], errors="coerce")
        revision_numerica.append({
            "tabla": nombre,
            "columna": columna,
            "vacios_o_no_numericos": int(valores.isna().sum()),
            
            # isinf detecta infinito positivo o negativo, que no es lo mismo que un dato vacío.
            "infinitos": int(np.isinf(valores).sum()),
        })
display(pd.DataFrame(revision_numerica))

,tabla,columna,faltantes,porcentaje
0,telemetria,datetime,0,0.0
1,telemetria,machineID,0,0.0
2,telemetria,volt,0,0.0
3,telemetria,rotate,0,0.0
4,telemetria,pressure,0,0.0
5,telemetria,vibration,0,0.0
6,errores,datetime,0,0.0
7,errores,machineID,0,0.0
8,errores,errorID,0,0.0
9,mantenimiento,datetime,0,0.0


,tabla,columna,vacios_o_no_numericos,infinitos
0,telemetria,machineID,0,0
1,telemetria,volt,0,0
2,telemetria,rotate,0,0
3,telemetria,pressure,0,0
4,telemetria,vibration,0,0
5,errores,machineID,0,0
6,mantenimiento,machineID,0,0
7,fallas,machineID,0,0
8,maquinas,machineID,0,0
9,maquinas,age,0,0


No se encontraron valores faltantes, valores no numéricos en las columnas revisadas ni infinitos. **Por ahora no hace falta rellenar datos.** Esto no significa que todas las mediciones sean físicamente normales: sus distribuciones y posibles valores extremos se revisarán en el EDA, usando el periodo de entrenamiento.

## 6. Distinguir duplicados de eventos simultáneos

Una fila idéntica repetida no es lo mismo que dos eventos diferentes en una misma hora.

- En telemetría esperamos una sola fila por máquina y hora.
- En errores puede haber distintos tipos de error en esa hora.
- En mantenimiento y fallas puede haber distintos componentes en esa hora.
- En el catálogo esperamos una fila por máquina.

Contaremos las filas repetidas y las repeticiones de cada clave. La cuenta representa filas adicionales después de la primera, no el número de grupos de eventos.

In [6]:
# La clave identifica un registro: en eventos incluye el tipo o componente para conservar eventos simultáneos.
claves_por_tabla = {
    "telemetria": ["machineID", "datetime"],
    "errores": ["machineID", "datetime", "errorID"],
    "mantenimiento": ["machineID", "datetime", "comp"],
    "fallas": ["machineID", "datetime", "failure"],
    "maquinas": ["machineID"],
}
revision_duplicados = []
for nombre, tabla in tablas.items():
    fila = {
        "tabla": nombre,
        # duplicated marca las filas posteriores a la primera; no elimina ninguna.
        "filas_identicas_repetidas": int(tabla.duplicated().sum()),
        
        # Comparar solo las columnas de la clave permite detectar registros que representan el mismo evento.
        "repeticiones_clave_completa": int(tabla.duplicated(claves_por_tabla[nombre]).sum()),
    }
    
    # Contar también coincidencias de máquina y hora, aunque correspondan a eventos diferentes.
    if "datetime" in tabla.columns:
        fila["repeticiones_maquina_hora"] = int(tabla.duplicated(["machineID", "datetime"]).sum())
    else:
        # En el catálogo no hay horas; NA indica que esta comprobación no aplica.
        fila["repeticiones_maquina_hora"] = pd.NA
    revision_duplicados.append(fila)
display(pd.DataFrame(revision_duplicados))

,tabla,filas_identicas_repetidas,repeticiones_clave_completa,repeticiones_maquina_hora
0,telemetria,0,0,0
1,errores,0,0,303
2,mantenimiento,0,0,758
3,fallas,0,0,42
4,maquinas,0,0,<NA>


No hay filas idénticas repetidas ni repeticiones de las claves completas. La telemetría tiene una única fila por máquina y hora.

En cambio, aparecen **303 filas adicionales de errores, 758 de mantenimiento y 42 de fallas** que comparten máquina y hora con otra fila. Como la clave completa no está repetida, corresponden a distintos tipos de error o componentes. **No debemos borrarlas como duplicados.** Más adelante habrá que contemplar fallas de varios componentes en una misma hora al definir la etiqueta multiclase; todavía no elegimos una regla.

## 7. Comprobar máquinas, categorías y antigüedad

Revisaremos que los identificadores tengan sentido y que cada máquina de los archivos esté en el catálogo. También comprobaremos que aparezcan únicamente los tipos de error, componentes y modelos esperados.

No estamos comparando frecuencias ni relaciones con fallas; solo comprobamos que los valores pertenezcan a las categorías previstas.

In [7]:
maquinas = tablas["maquinas"]

# Crear el conjunto de máquinas válidas para revisar los otros archivos.
ids_catalogo = set(maquinas["machineID"])
revision_maquinas = []
for nombre, tabla in tablas.items():
    ids = tabla["machineID"]
    ids_numericos = pd.to_numeric(ids, errors="coerce")
    
    # El símbolo | significa “o”: marcar vacíos, IDs no positivos o números con parte decimal.
    ids_invalidos = ids_numericos.isna() | (ids_numericos <= 0) | (ids_numericos % 1 != 0)
    revision_maquinas.append({
        "tabla": nombre,
        "maquinas_distintas": ids.nunique(),
        "IDs_invalidos": int(ids_invalidos.sum()),
        
        # La resta encuentra máquinas que aparecen en los eventos pero no en el catálogo.
        "IDs_sin_catalogo": sorted(set(ids) - ids_catalogo),
        
        # La resta inversa encuentra máquinas del catálogo sin registros en esta fuente.
        "IDs_catalogo_sin_registros": sorted(ids_catalogo - set(ids)),
    })
display(pd.DataFrame(revision_maquinas))

# Guardar la columna y sus categorías válidas para cada tipo de evento.
categorias_esperadas = {
    "errores": ("errorID", {"error1", "error2", "error3", "error4", "error5"}),
    "mantenimiento": ("comp", {"comp1", "comp2", "comp3", "comp4"}),
    "fallas": ("failure", {"comp1", "comp2", "comp3", "comp4"}),
    "maquinas": ("model", {"model1", "model2", "model3", "model4"}),
}
revision_categorias = []

# Separar el nombre de la tabla, la columna revisada y sus categorías permitidas.
for nombre, (columna, esperadas) in categorias_esperadas.items():
    
    # Tomar categorías distintas sin vacíos; no contar aquí cuántas veces se repiten.
    observadas = set(tablas[nombre][columna].dropna())
    revision_categorias.append({
        "tabla": nombre,
        "columna": columna,
        "valores_observados": sorted(observadas),
        "valores_no_esperados": sorted(observadas - esperadas),
    })
display(pd.DataFrame(revision_categorias))

# Revisar la antigüedad como número; cero años es válido y no se trata como faltante.
edades = pd.to_numeric(maquinas["age"], errors="coerce")
print("Antigüedad mínima:", edades.min(), "años")
print("Antigüedad máxima:", edades.max(), "años")
print("Antigüedades negativas o no numéricas:", int((edades.isna() | (edades < 0)).sum()))

,tabla,maquinas_distintas,IDs_invalidos,IDs_sin_catalogo,IDs_catalogo_sin_registros
0,telemetria,100,0,[],[]
1,errores,100,0,[],[]
2,mantenimiento,100,0,[],[]
3,fallas,98,0,[],"[6, 77]"
4,maquinas,100,0,[],[]


,tabla,columna,valores_observados,valores_no_esperados
0,errores,errorID,"[error1, error2, error3, error4, error5]",[]
1,mantenimiento,comp,"[comp1, comp2, comp3, comp4]",[]
2,fallas,failure,"[comp1, comp2, comp3, comp4]",[]
3,maquinas,model,"[model1, model2, model3, model4]",[]


Antigüedad mínima: 0 años
Antigüedad máxima: 20 años
Antigüedades negativas o no numéricas: 0


La telemetría, los errores y el mantenimiento contienen registros de las **100 máquinas del catálogo**. El archivo de fallas contiene **98 máquinas**: los identificadores **6 y 77** no aparecen en ese archivo. Esto indica ausencia de fallas registradas para esas máquinas, no un error de relación ni una razón para eliminarlas. No hay identificadores desconocidos ni inválidos. Los tipos de error, componentes y modelos coinciden con los esperados. La antigüedad va de **0 a 20 años**, sin valores negativos; cero años no es, por sí solo, un dato faltante.

## 8. Comprobar la continuidad horaria de la telemetría

Ordenaremos una copia de la telemetría por máquina y fecha. Para cada máquina compararemos las horas observadas con una cuadrícula horaria que va desde el inicio hasta el final de la cobertura global.

Esta revisión detecta huecos internos y máquinas que empiezan tarde o terminan antes que las demás. También contaremos los saltos entre registros que sean distintos de una hora. Los conteos solo revisan disponibilidad de datos, sin estudiar patrones de los sensores.

In [8]:
# Ordenar una copia por máquina y fecha para calcular diferencias entre horas consecutivas.
telemetria = tablas["telemetria"].sort_values(["machineID", "datetime"]).copy()
inicio_cobertura = telemetria["datetime"].min()
fin_cobertura = telemetria["datetime"].max()

# Crear todas las horas entre los extremos de la cobertura, incluidos ambos extremos.
horas_esperadas = pd.date_range(inicio_cobertura, fin_cobertura, freq="h")

revision_continuidad = []

# groupby separa los registros de cada máquina para revisar su continuidad por separado.
for maquina_id, grupo in telemetria.groupby("machineID"):
    horas_observadas = pd.DatetimeIndex(grupo["datetime"])
    
    # difference devuelve las horas esperadas que no aparecen entre las observadas.
    horas_faltantes = horas_esperadas.difference(horas_observadas)
    
    # La comparación inversa detecta registros que no pertenecen a la cuadrícula horaria.
    horas_fuera_cuadricula = horas_observadas.difference(horas_esperadas)
    
    # diff resta la fecha anterior; la primera queda vacía porque no tiene una anterior.
    diferencias = grupo["datetime"].diff().dropna()
    revision_continuidad.append({
        "machineID": maquina_id,
        "registros": len(grupo),
        "primera_fecha": grupo["datetime"].min(),
        "ultima_fecha": grupo["datetime"].max(),
        "horas_faltantes": len(horas_faltantes),
        "horas_fuera_cuadricula": len(horas_fuera_cuadricula),
        "saltos_distintos_de_1h": int((diferencias != pd.Timedelta(hours=1)).sum()),
    })

continuidad = pd.DataFrame(revision_continuidad)
print("Horas esperadas por máquina:", len(horas_esperadas))
print("Máquinas revisadas:", len(continuidad))
print("Total de horas faltantes:", continuidad["horas_faltantes"].sum())
print("Total de saltos distintos de una hora:", continuidad["saltos_distintos_de_1h"].sum())

# Mostrar diez máquinas como ejemplo; los conteos anteriores usan todas las máquinas.
display(continuidad.head(10))

# Filtrar solo máquinas con al menos un problema; | combina las tres condiciones con “o”.
problemas_continuidad = continuidad[
    (continuidad["horas_faltantes"] > 0)
    | (continuidad["horas_fuera_cuadricula"] > 0)
    | (continuidad["saltos_distintos_de_1h"] > 0)
]
print("Máquinas con problemas de continuidad:", len(problemas_continuidad))

# Mostrar el detalle únicamente si se encontró alguna máquina con problemas.
if not problemas_continuidad.empty:
    display(problemas_continuidad)

Horas esperadas por máquina: 8761
Máquinas revisadas: 100
Total de horas faltantes: 0
Total de saltos distintos de una hora: 0


,machineID,registros,primera_fecha,ultima_fecha,horas_faltantes,horas_fuera_cuadricula,saltos_distintos_de_1h
0,1,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
1,2,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
2,3,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
3,4,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
4,5,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
5,6,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
6,7,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
7,8,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
8,9,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0
9,10,8761,2015-01-01 06:00:00,2016-01-01 06:00:00,0,0,0


Máquinas con problemas de continuidad: 0


Cada una de las **100 máquinas tiene 8,761 mediciones horarias**, sin huecos ni saltos diferentes de una hora. El total coincide: **100 × 8,761 = 876,100 filas**.

Las 8,761 horas incluyen ambos extremos del intervalo, desde las 06:00 del 1 de enero de 2015 hasta las 06:00 del 1 de enero de 2016. No hace falta inventar horas anteriores al inicio ni rellenar huecos dentro de esta cobertura.

## 9. Cierre de la revisión

**Lo que quedó comprobado**

- Los cinco archivos se leen correctamente y tienen las columnas esperadas.
- No se encontraron valores faltantes, fechas inválidas, infinitos ni filas idénticas repetidas.
- Los registros se relacionan con las 100 máquinas del catálogo.
- La telemetría es continua por máquina y tiene una fila por hora.

**Lo que debemos tener presente al continuar**

- Conservar los eventos simultáneos de distintos tipos o componentes; no son duplicados.
- Resolver explícitamente cómo representar varias fallas dentro del horizonte cuando construyamos la etiqueta multiclase.
- Conservar por ahora el historial de mantenimiento de 2014 y documentar cómo se usará.
- Respetar las fechas de la partición temporal, aunque los CSV incluyan horas de enero de 2016.
- Revisar después la relación entre mantenimiento y fallas para evitar que información posterior a una falla entre como predictor.

Esta revisión no modificó los CSV, no creó conjuntos procesados y no entrenó modelos. Las explicaciones de resultados corresponden a los archivos originales revisados; si se reemplazan los CSV, hay que volver a ejecutar todo y actualizar esas explicaciones cuando cambien las salidas.